<a href="https://colab.research.google.com/github/AndresMontesDeOca/vision_computadora_II/blob/VpC2_2026/CS_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Clasificación y Detección de Objetos en Counter-Strike**

Este notebook contiene el pipeline completo para descargar un dataset de Counter-Strike desde Roboflow, realizar análisis exploratorio de datos (EDA), entrenar un modelo YOLOv11 de Ultralytics, evaluar las métricas de rendimiento y realizar inferencias/predicciones visuales utilizando Google Drive como almacenamiento persistente.

## **1. Configuración del Entorno e Importación de Librerías**

En esta sección instalamos las dependencias requeridas, montamos Google Drive para persistencia de datos e importamos los módulos necesarios.

In [ ]:
# Instalar librerías necesarias si no están presentes
# !pip install roboflow ultralytics

import os
import yaml
import json
import random
import cv2
import glob
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import userdata
from google.colab import drive
from roboflow import Roboflow
from ultralytics import YOLO

# Montar Google Drive al inicio de la sesión
drive.mount('/content/drive')

## **2. Descarga y Verificación del Dataset**

Utilizamos la API de Roboflow para descargar la versión adaptada a YOLOv11 de nuestro dataset de Counter-Strike de manera automatizada.

In [ ]:
# Ruta esperada del dataset
dataset_path = '/content/Counter-Strike-1'

# Verificar si el dataset ya existe en el entorno
if not os.path.exists(dataset_path):
    print("El dataset no fue detectado. Iniciando descarga desde Roboflow...")
    api_key = userdata.get("ROBOFLOW_API_KEY")
    rf = Roboflow(api_key=api_key)

    # Descarga del dataset en formato YOLOv11
    project = rf.workspace("counter-strike-uovxc").project("counter-strike-h9y80")
    version = project.version(1)
    dataset = version.download("yolov11")
else:
    print(f"El dataset ya existe en '{dataset_path}'. Se omitió la descarga para optimizar tiempo.")

## **3. Análisis Exploratorio de Datos (EDA)**

Definimos funciones de visualización para inspeccionar nuestras imágenes de entrenamiento, analizar las distribuciones de clases y resoluciones para asegurar un entrenamiento estable.

In [ ]:
# Cargar y mostrar la configuración de clases y rutas
data_yaml_path = '/content/Counter-Strike-1/data.yaml'
with open(data_yaml_path, 'r') as file:
    dataset_info = yaml.safe_load(file)

print("Estructura y clases del dataset:")
print(json.dumps(dataset_info, indent=4))
print("\n" + "="*40 + "\n")

# --- Conteo de Imágenes por Carpeta ---
dataset_root = '/content/Counter-Strike-1'
for split in ['train', 'val', 'test']:
    yaml_key = split
    if split == 'val' and 'val' not in dataset_info and 'valid' in dataset_info:
        yaml_key = 'valid'

    if yaml_key in dataset_info:
        split_path = dataset_info[yaml_key].lstrip('./').replace('../', '')
        full_split_path = os.path.abspath(os.path.join(dataset_root, split_path))

        image_extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.JPG', '*.JPEG', '*.PNG')
        images_found = []
        for ext in image_extensions:
            images_found.extend(glob.glob(os.path.join(full_split_path, ext)))

        print(f"Carpeta '{split}': {len(images_found)} imágenes encontradas en {full_split_path}")
    else:
        print(f"No se encontró la definición de '{split}' o '{yaml_key}' en data.yaml")
print("\n" + "="*40 + "\n")

# Definición de parámetros principales para el EDA
images_dir = '/content/Counter-Strike-1/train/images'
labels_dir = '/content/Counter-Strike-1/train/labels'
class_names = dataset_info.get('names', ['ct', 't'])

# Función para graficar muestras YOLO
def plot_yolo_sample(img_path, lbl_path, class_names):
    if not os.path.exists(img_path):
        return
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w, _ = image.shape
    colors = [(0, 0, 255), (255, 0, 0)]  # Azul para CT, Rojo para Terroristas

    if os.path.exists(lbl_path):
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls_id = int(parts[0])
                    x_center, y_center, bbox_w, bbox_h = map(float, parts[1:])
                    x1 = int((x_center - bbox_w / 2) * w)
                    y1 = int((y_center - bbox_h / 2) * h)
                    x2 = int((x_center + bbox_w / 2) * w)
                    y2 = int((y_center + bbox_h / 2) * h)
                    color = colors[cls_id % len(colors)]
                    cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
                    label = class_names[cls_id]
                    cv2.putText(image, label, (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

def plot_random_samples(images_dir, labels_dir, class_names, num_samples=3):
    all_images = [f for f in os.listdir(images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    selected_images = random.sample(all_images, min(len(all_images), num_samples))
    for img_name in selected_images:
        lbl_name = img_name.rsplit('.', 1)[0] + '.txt'
        plot_yolo_sample(os.path.join(images_dir, img_name), os.path.join(labels_dir, lbl_name), class_names)

# Mostrar muestras aleatorias etiquetadas
plot_random_samples(images_dir, labels_dir, class_names, num_samples=2)

In [ ]:
# Análisis del balance de clases en cada partición
particiones = {
    'Train': '/content/Counter-Strike-1/train/labels',
    'Valid': '/content/Counter-Strike-1/valid/labels',
    'Test': '/content/Counter-Strike-1/test/labels'
}

resultados = {}
for nombre, ruta in particiones.items():
    counts = {0: 0, 1: 0}
    if os.path.exists(ruta):
        for file in os.listdir(ruta):
            if file.endswith('.txt'):
                with open(os.path.join(ruta, file), 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if parts:
                            cls_id = int(parts[0])
                            if cls_id in counts:
                                counts[cls_id] += 1
    resultados[nombre] = counts

df_balance = pd.DataFrame(resultados).rename(index={0: 'Counter-Terrorist (ct)', 1: 'Terrorist (t)'})
display(df_balance)

# Graficar Balance de Clases
fig, ax = plt.subplots(figsize=(8, 5))
df_balance.T.plot(kind='bar', ax=ax, color=['royalblue', 'crimson'])
plt.title('Balanceo de Clases por Partición')
plt.ylabel('Cantidad de Instancias')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## **4. Entrenamiento del Modelo (YOLOv11)**

Inicializamos y entrenamos el detector con arquitectura YOLOv11 utilizando los hiperparámetros óptimos seleccionados.

In [ ]:
# Configuración del flujo de entrenamiento
entrenar = True  # True para entrenar un modelo nuevo, False para saltarlo y cargar pesos previos

# Versión de YOLOv11 a utilizar. Opciones disponibles:
# - "yolo11n.pt" (Nano: extremadamente rápido, ideal para CPU/móviles)
# - "yolo11s.pt" (Small: balance velocidad/precisión)
# - "yolo11m.pt" (Medium: para mayor precisión en detecciones complejas)
# - "yolo11l.pt" (Large: mayor precisión, requiere más memoria GPU)
# - "yolo11x.pt" (X-Large: máxima precisión, entrenamiento más lento)
modelo_version = "yolo11n.pt"

# Hiperparámetros generales de entrenamiento
epocas = 200
tamanio_batch = 8
paciencia = 15  # Número de épocas a esperar sin mejora antes de detener el entrenamiento (Early Stopping)

print(f"Configuración cargada:\n- Entrenar: {entrenar}\n- Modelo base: {modelo_version}\n- Épocas: {epocas}\n- Batch size: {tamanio_batch}\n- Paciencia (Early Stopping): {paciencia}")

In [ ]:
if entrenar:
    print(f"Iniciando entrenamiento del modelo seleccionado: {modelo_version}...")
    # Inicializar la arquitectura elegida dinámicamente
    model = YOLO(modelo_version)

    # Comenzar entrenamiento con los parámetros seleccionados por el usuario
    results = model.train(
        data="/content/Counter-Strike-1/data.yaml",
        epochs=epocas,
        patience=paciencia,
        imgsz=640,
        batch=tamanio_batch,
        project="runs",
        name="cs_v2",
        exist_ok=True
    )
else:
    print("\n[INFO] 'entrenar' está desactivado. Se omite el entrenamiento de esta sesión.")
    print("Los pesos y métricas se restaurarán o leerán directamente desde Google Drive en la sección 5.")

## **5. Respaldo y Recuperación de Progreso (Métricas y Pesos)**

Esta celda actúa de manera inteligente. Si acabas de entrenar, subirá las métricas y tus mejores pesos (`best.pt`) a Google Drive de manera persistente. En caso de reiniciar el entorno de ejecución, recuperará automáticamente estos archivos desde Drive.

In [ ]:
import os
import shutil
import pandas as pd

# Obtener el nombre base del modelo limpio (ej: "yolo11n" desde "yolo11n.pt")
modelo_id = modelo_version.replace(".pt", "")

# Definición de directorios locales de origen
save_dir = "runs/detect/runs/cs_v2"
results_csv_path = os.path.join(save_dir, "results.csv")
local_weights_path = os.path.join(save_dir, "weights", "best.pt")

# Definición de directorios y nombres de archivos dinámicos en Drive y Local
drive_dest_dir = '/content/drive/MyDrive/CounterStrike_YOLO_Metrics'

# Nombres de archivos con la versión del modelo incorporada (Aislamiento de modelos)
csv_output_name = f"{modelo_id}_training_metrics_backup.csv"
weights_output_name = f"{modelo_id}_best.pt"

# Rutas de destino locales y en Drive
local_csv_backup_path = os.path.join(save_dir, csv_output_name)
drive_csv_path = os.path.join(drive_dest_dir, csv_output_name)

drive_weights_dir = os.path.join(drive_dest_dir, "weights")
drive_weights_path = os.path.join(drive_weights_dir, weights_output_name)
local_target_weights_path = os.path.join(save_dir, "weights", weights_output_name)

os.makedirs(save_dir, exist_ok=True)

print(f"[INFO] Buscando/Guardando archivos específicos para la arquitectura activa: '{modelo_id}'")
print(f"[INFO] Los demás modelos guardados en tu Drive no serán modificados ni leídos en esta sesión.\n")

# Lógica inteligente de sincronización dinámica por modelo
if os.path.exists(results_csv_path):
    print(f"¡Se detectó un nuevo entrenamiento local! Guardando métricas y pesos para {modelo_id}...")
    df_temp = pd.read_csv(results_csv_path)
    df_temp.to_csv(local_csv_backup_path, index=False)

    try:
        os.makedirs(drive_dest_dir, exist_ok=True)
        shutil.copy(local_csv_backup_path, drive_csv_path)
        print(f"Métricas de {modelo_id} respaldadas en Drive con éxito.")
    except Exception as e:
        print(f"Error al respaldar métricas: {e}")

    if os.path.exists(local_weights_path):
        try:
            os.makedirs(drive_weights_dir, exist_ok=True)
            # Renombrar y copiar los pesos locales al Drive con su nombre de versión
            shutil.copy(local_weights_path, drive_weights_path)
            # También guardamos una copia con el nombre específico localmente para inferencia
            shutil.copy(local_weights_path, local_target_weights_path)
            print(f"Pesos respaldados con éxito en Drive como '{weights_output_name}'.")
        except Exception as e:
            print(f"Error al respaldar pesos: {e}")

elif os.path.exists(drive_csv_path):
    print(f"¡Se detectó un respaldo en Google Drive para el modelo {modelo_id}! Restaurando...")
    try:
        shutil.copy(drive_csv_path, local_csv_backup_path)
        print(f"Métricas de {modelo_id} restauradas localmente.")
    except Exception as e:
        print(f"Error al restaurar CSV: {e}")

    if os.path.exists(drive_weights_path):
        try:
            os.makedirs(os.path.dirname(local_weights_path), exist_ok=True)
            # Restaurar el archivo en ambas rutas (con nombre específico y el genérico para compatibilidad)
            shutil.copy(drive_weights_path, local_target_weights_path)
            shutil.copy(drive_weights_path, local_weights_path)
            print(f"Pesos de '{weights_output_name}' restaurados localmente para inferencia inmediata.")
        except Exception as e:
            print(f"Error al restaurar pesos: {e}")
else:
    print(f"No se encontraron métricas locales ni en Google Drive para el modelo {modelo_id}.")

# Cargar df final si existe
if os.path.exists(local_csv_backup_path):
    df = pd.read_csv(local_csv_backup_path)
    display(df.head(5))
else:
    df = None

## **6. Visualización del Rendimiento del Entrenamiento**

Graficamos el progreso del modelo a través de las diferentes épocas analizando las curvas de pérdidas (*loss*) y de precisión (*mAP*).

In [ ]:
if df is not None:
    # Obtener el nombre limpio del modelo (ej: YOLOv11n)
    nombre_modelo_grafico = f"YOLOv11 ({modelo_id})" if 'modelo_id' in globals() else "YOLOv11"

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Título general para toda la figura
    fig.suptitle(f"Métricas de Rendimiento - {nombre_modelo_grafico}", fontsize=16, fontweight='bold', y=0.98)

    # --- Gráfico 1: Evolución de Pérdidas (Dos colores base con tonalidades oscura/clara) ---
    # Color Base 1 (Box Loss): Azul Oscuro (train) y Azul Claro (val)
    # Color Base 2 (Class Loss): Rojo Oscuro (train) y Rojo Claro/Rosa (val)
    axes[0].plot(df["epoch"], df["train/box_loss"], label="train/box_loss (Box Train)", color='#0b559f', linewidth=2)      # Azul Oscuro
    axes[0].plot(df["epoch"], df["val/box_loss"], label="val/box_loss (Box Val)", color='#89bedc', linewidth=2, linestyle='--') # Azul Claro
    axes[0].plot(df["epoch"], df["train/cls_loss"], label="train/cls_loss (Class Train)", color='#b30000', linewidth=2)     # Rojo Oscuro
    axes[0].plot(df["epoch"], df["val/cls_loss"], label="val/cls_loss (Class Val)", color='#ff9999', linewidth=2, linestyle='--') # Rojo Claro (Rosa)

    axes[0].set_xlabel("Época")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Evolución de Pérdidas")
    axes[0].legend()
    axes[0].grid(True, linestyle=':', alpha=0.6)

    # --- Gráfico 2: Evolución de mAP ---
    axes[1].plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP50", color='green', linewidth=2)
    axes[1].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95", color='orange', linewidth=2)
    axes[1].set_xlabel("Época")
    axes[1].set_ylabel("mAP")
    axes[1].set_title("Evolución del mAP en Validación")
    axes[1].legend()
    axes[1].grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()
else:
    print("No hay datos de entrenamiento listos para graficar.")

## **7. Inferencia y Predicciones en Nuevas Muestras**

Esta sección carga los pesos optimizados del entrenamiento (`best.pt`) y realiza una prueba de predicción visual sobre una imagen aleatoria.

In [ ]:
# 1. Selección de imagen al azar
train_images_dir = "/content/Counter-Strike-1/test/images"
if os.path.exists(train_images_dir):
    all_train_images = [f for f in os.listdir(train_images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    random_img_name = random.choice(all_train_images)
    img_path = os.path.join(train_images_dir, random_img_name)
else:
    raise FileNotFoundError("Directorio de entrenamiento no encontrado.")

# 2. Cargar el mejor modelo disponible
local_weights_path = "runs/detect/runs/cs_v2/weights/best.pt"
drive_weights_path = "/content/drive/MyDrive/CounterStrike_YOLO_Metrics/weights/best.pt"

if os.path.exists(local_weights_path):
    print(f"Cargando pesos optimizados locales: {local_weights_path}")
    inference_model = YOLO(local_weights_path)
elif os.path.exists(drive_weights_path):
    print(f"Cargando pesos optimizados de Drive: {drive_weights_path}")
    inference_model = YOLO(drive_weights_path)
else:
    print("Cargando modelo preentrenado base para demostración rápida.")
    inference_model = YOLO("yolo11n.pt")

# 3. Realizar predicción
preds = inference_model.predict(source=img_path, imgsz=640, conf=0.4, save=True, project="runs", name="cs_v2_pred", exist_ok=True)

# 4. Mostrar el resultado visual final
result_image_path = os.path.join(preds[0].save_dir, os.path.basename(preds[0].path))
if os.path.exists(result_image_path):
    img = cv2.imread(result_image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

In [ ]:
import os
import json
import shutil
from ultralytics import YOLO

# 1. Definir rutas de los pesos óptivos guardados e identificador del modelo
modelo_id = modelo_version.replace(".pt", "") if 'modelo_version' in globals() else "yolo11n"
local_weights_path = "runs/detect/runs/cs_v2/weights/best.pt"
drive_weights_path = f"/content/drive/MyDrive/CounterStrike_YOLO_Metrics/weights/{modelo_id}_best.pt"

# Determinar cuál archivo de pesos cargar
if os.path.exists(local_weights_path):
    path_a_usar = local_weights_path
    print(f"[INFO] Cargando pesos locales para evaluación: {local_weights_path}")
elif os.path.exists(drive_weights_path):
    path_a_usar = drive_weights_path
    print(f"[INFO] Cargando pesos respaldados en Drive para evaluación: {drive_weights_path}")
else:
    path_a_usar = "yolo11n.pt"
    print("[ALERTA] No se detectaron pesos entrenados. Se evaluará el modelo base preentrenado.")

# 2. Cargar el modelo YOLO con los mejores pesos
model_eval = YOLO(path_a_usar)

# 3. Ejecutar la validación específicamente en el conjunto de prueba (split='test')
print(f"\n--- Iniciando Evaluación en el Conjunto de Test para {modelo_id} ---")
metrics_test = model_eval.val(
    data="/content/Counter-Strike-1/data.yaml",
    split="test",
    imgsz=640,
    project="runs",
    name="cs_v2_test_evaluation",
    exist_ok=True
)

# 4. Extraer métricas de interés
test_results = {
    "modelo_id": modelo_id,
    "mAP50": float(metrics_test.results_dict['metrics/mAP50(B)']),
    "mAP50-95": float(metrics_test.results_dict['metrics/mAP50-95(B)']),
    "precision": float(metrics_test.results_dict['metrics/precision(B)']),
    "recall": float(metrics_test.results_dict['metrics/recall(B)']),
    "ct_mAP50": float(metrics_test.results_dict.get('metrics/mAP50(B)', 0.0)) if len(metrics_test.curves_results) == 0 else float(metrics_test.box.class_result(0)[2]),
    "t_mAP50": float(metrics_test.results_dict.get('metrics/mAP50(B)', 0.0)) if len(metrics_test.curves_results) == 0 else float(metrics_test.box.class_result(1)[2])
}

# 5. Guardar las métricas de test en un archivo JSON local y respaldar en Google Drive
drive_dest_dir = '/content/drive/MyDrive/CounterStrike_YOLO_Metrics'
json_file_name = f"{modelo_id}_test_metrics_backup.json"
local_json_path = os.path.join("runs/detect/runs/cs_v2", json_file_name)
drive_json_path = os.path.join(drive_dest_dir, json_file_name)

# Guardar archivo local
os.makedirs(os.path.dirname(local_json_path), exist_ok=True)
with open(local_json_path, 'w') as f:
    json.dump(test_results, f, indent=4)

# Copiar a Drive de manera segura
try:
    os.makedirs(drive_dest_dir, exist_ok=True)
    shutil.copy(local_json_path, drive_json_path)
    print(f"\n[OK] Métricas de test de '{modelo_id}' respaldadas en Drive con éxito: {drive_json_path}")
except Exception as e:
    print(f"\n[ALERTA] Error al guardar el archivo de métricas de test en Google Drive: {e}")

# 6. Mostrar de forma clara y formateada las métricas clave obtenidas
print("\n" + "="*50)
print("     MÉTRICAS FINALES EN EL CONJUNTO DE TEST")
print("="*50)
print(f"mAP50 (Box):      {test_results['mAP50']:.4f}")
print(f"mAP50-95 (Box):   {test_results['mAP50-95']:.4f}")
print(f"Precisión (P):    {test_results['precision']:.4f}")
print(f"Exhaustividad (R): {test_results['recall']:.4f}")
print("="*50)